In [ ]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import kernel_g
import polars as pl

START_DATE = pd.to_datetime('2023-01-01')
END_DATE = pd.to_datetime('2025-11-15')
date_range = pd.bdate_range(START_DATE, END_DATE)
STATISTIC_COL = 'top_5_z_score_abs'

bad_dates = []
inputs = []
dfs = []
for date in tqdm(date_range):
    #print(date)
    is_there_trade_file = Path(f"/Volumes/T7/data/trades/time_partioned/{date.strftime('%Y%m%d')}.parquet").exists()
    if not is_there_trade_file:
        continue
    df = pl.read_parquet(f"/Volumes/T7/alpha/10_15_full/{date.strftime('%Y%m%d')}.parquet")

    df_r = pl.read_parquet(f"/Volumes/T7/alpha/10_15_full/{date.strftime('%Y%m%d')}_R.parquet")

    df = kernel_g.kernel_stats_pl(df)
    f_unique = df.unique(subset=['kernel'])
    f_unique = f_unique.to_pandas()

    r = kernel_g.kernel_stats_pl(df_r)
    r_unique = r.unique(subset=['kernel'])
    r_unique = r_unique.to_pandas()

    lower_threshold = r_unique[STATISTIC_COL].quantile(0.99)

    f_unique = f_unique[f_unique[STATISTIC_COL]>lower_threshold]

    df = df.filter(pl.col(STATISTIC_COL)>lower_threshold)
    df = df.with_columns(pl.col("z_score_abs").max().over('ticker').alias('max_z_score_abs_ticker'))
    df = df.with_columns(pl.col("z_score").max().over('ticker').alias('max_z_score_ticker'))
    df = df.with_columns(pl.col("z_score").min().over('ticker').alias('min_z_score_ticker'))
    df = df.filter(pl.col("z_score_abs")==pl.col("max_z_score_abs_ticker"))
    #df = df.group_by(['date', 'ticker']).agg(pl.col(STATISTIC_COL).max().alias("selected_ticker"))

    df = df.with_columns(pl.lit(date).alias('date'))
    dfs.append(df)

In [ ]:
df = pl.concat(dfs)

In [ ]:
df.write_parquet("/Volumes/T7/alpha/10_15_max_ticker.parquet")

# Version without 2_7

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import kernel_g
import polars as pl

START_DATE = pd.to_datetime('2023-01-01')
END_DATE = pd.to_datetime('2025-11-15')
date_range = pd.bdate_range(START_DATE, END_DATE)
STATISTIC_COL = 'top_5_z_score_abs'

bad_dates = []
inputs = []
dfs = []
for date in tqdm(date_range):
    #print(date)
    is_there_trade_file = Path(f"/Volumes/T7/data/trades/time_partioned/{date.strftime('%Y%m%d')}.parquet").exists()
    if not is_there_trade_file:
        continue
    df = pl.read_parquet(f"/Volumes/T7/alpha/10_15_full/{date.strftime('%Y%m%d')}.parquet")

    df_r = pl.read_parquet(f"/Volumes/T7/alpha/10_15_full/{date.strftime('%Y%m%d')}_R.parquet")

    df = kernel_g.kernel_stats_pl(df)
    f_unique = df.unique(subset=['kernel'])
    f_unique = f_unique.to_pandas()

    r = kernel_g.kernel_stats_pl(df_r)
    r_unique = r.unique(subset=['kernel'])
    r_unique = r_unique.to_pandas()

    lower_threshold = r_unique[STATISTIC_COL].quantile(0.99)

    f_unique = f_unique[f_unique[STATISTIC_COL] > lower_threshold]

    df = df.filter(pl.col(STATISTIC_COL) > lower_threshold)
    df = df.with_columns(pl.when((pl.col('kernel') == '2_7')|(pl.col('kernel') == '2_8')|(pl.col('kernel')=='2_9'))
        .then(1)
        .otherwise(0)
        .alias("is_2_7")
    )
    df = df.filter(pl.col('is_2_7')==0)
    df = df.with_columns(pl.col("z_score_abs").max().over('ticker').alias('max_z_score_abs_ticker'))
    df = df.filter(pl.col("z_score_abs") == pl.col("max_z_score_abs_ticker"))
    #df = df.group_by(['date', 'ticker']).agg(pl.col(STATISTIC_COL).max().alias("selected_ticker"))

    df = df.with_columns(pl.lit(date).alias('date'))
    dfs.append(df)


In [ ]:
df = pl.concat(dfs)
df.write_parquet("/Volumes/T7/alpha/10_15_max_ticker_no_2_7.parquet")

# Version with 0

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import kernel_g
import polars as pl

START_DATE = pd.to_datetime('2023-01-01')
END_DATE = pd.to_datetime('2025-11-15')
date_range = pd.bdate_range(START_DATE, END_DATE)
STATISTIC_COL = 'top_5_z_score_abs'

bad_dates = []
inputs = []
dfs = []
for date in tqdm(date_range):
    #print(date)
    is_there_trade_file = Path(f"/Volumes/T7/data/trades/time_partioned/{date.strftime('%Y%m%d')}.parquet").exists()
    if not is_there_trade_file:
        continue
    df = pl.read_parquet(f"/Volumes/T7/alpha/10_15_full/{date.strftime('%Y%m%d')}.parquet")

    df_r = pl.read_parquet(f"/Volumes/T7/alpha/10_15_full/{date.strftime('%Y%m%d')}_R.parquet")

    df = kernel_g.kernel_stats_pl(df)
    f_unique = df.unique(subset=['kernel'])
    f_unique = f_unique.to_pandas()

    r = kernel_g.kernel_stats_pl(df_r)
    r_unique = r.unique(subset=['kernel'])
    r_unique = r_unique.to_pandas()

    lower_threshold = r_unique[STATISTIC_COL].quantile(0.99)

    f_unique = f_unique[f_unique[STATISTIC_COL] > lower_threshold]

    df = df.filter(pl.col(STATISTIC_COL) > lower_threshold)
    df = df.filter(pl.col('grid_mod_n')!=0)
    df = df.with_columns(pl.col("z_score_abs").max().over('ticker').alias('max_z_score_abs_ticker'))
    df = df.filter(pl.col("z_score_abs") == pl.col("max_z_score_abs_ticker"))
    #df = df.group_by(['date', 'ticker']).agg(pl.col(STATISTIC_COL).max().alias("selected_ticker"))

    df = df.with_columns(pl.lit(date).alias('date'))
    dfs.append(df)


In [ ]:
df = pl.concat(dfs)
df.write_parquet("/Volumes/T7/alpha/10_15_max_ticker_no_0.parquet")

# Stock level max but 60 second freqency

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import kernel_g
import polars as pl

START_DATE = pd.to_datetime('2023-01-01')
END_DATE = pd.to_datetime('2025-11-15')
date_range = pd.bdate_range(START_DATE, END_DATE)
STATISTIC_COL = 'top_5_z_score_abs'

bad_dates = []
inputs = []
dfs = []
for date in tqdm(date_range):
    #print(date)
    is_there_trade_file = Path(f"/Volumes/T7/data/trades/time_partioned/{date.strftime('%Y%m%d')}.parquet").exists()
    if not is_there_trade_file:
        continue
    df = pl.read_parquet(f"/Volumes/T7/alpha/10_15_60s_full/{date.strftime('%Y%m%d')}.parquet")

    df_r = pl.read_parquet(f"/Volumes/T7/alpha/10_15_60s_full/{date.strftime('%Y%m%d')}_R.parquet")

    df = kernel_g.kernel_stats_pl(df)
    f_unique = df.unique(subset=['kernel'])
    f_unique = f_unique.to_pandas()

    r = kernel_g.kernel_stats_pl(df_r)
    r_unique = r.unique(subset=['kernel'])
    r_unique = r_unique.to_pandas()

    lower_threshold = r_unique[STATISTIC_COL].quantile(0.99)

    f_unique = f_unique[f_unique[STATISTIC_COL]>lower_threshold]

    df = df.filter(pl.col(STATISTIC_COL)>lower_threshold)
    df = df.with_columns(pl.col("z_score_abs").max().over('ticker').alias('max_z_score_abs_ticker'))
    df = df.filter(pl.col("z_score_abs")==pl.col("max_z_score_abs_ticker"))
    #df = df.group_by(['date', 'ticker']).agg(pl.col(STATISTIC_COL).max().alias("selected_ticker"))

    df = df.with_columns(pl.lit(date).alias('date'))
    dfs.append(df)

In [ ]:
df = pl.concat(dfs)
df.write_parquet("/Volumes/T7/alpha/10_15_60s_max_ticker.parquet")

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import kernel_g
import polars as pl

START_DATE = pd.to_datetime('2023-01-01')
END_DATE = pd.to_datetime('2025-11-15')
date_range = pd.bdate_range(START_DATE, END_DATE)
STATISTIC_COL = 'max_z_score_abs'

bad_dates = []
inputs = []
dfs = []
for date in tqdm(date_range):
    #print(date)
    is_there_trade_file = Path(f"/Volumes/T7/data/trades/time_partioned/{date.strftime('%Y%m%d')}.parquet").exists()
    if not is_there_trade_file:
        continue
    df = pl.read_parquet(f"/Volumes/T7/alpha/10_15_60s_full/{date.strftime('%Y%m%d')}.parquet")

    df_r = pl.read_parquet(f"/Volumes/T7/alpha/10_15_60s_full/{date.strftime('%Y%m%d')}_R.parquet")

    df = kernel_g.kernel_stats_pl(df)
    f_unique = df.unique(subset=['kernel'])
    f_unique = f_unique.to_pandas()

    r = kernel_g.kernel_stats_pl(df_r)
    r_unique = r.unique(subset=['kernel'])
    r_unique = r_unique.to_pandas()

    lower_threshold = r_unique[STATISTIC_COL].quantile(0.99)

    f_unique = f_unique[f_unique[STATISTIC_COL]>lower_threshold]

    df = df.filter(pl.col(STATISTIC_COL)>lower_threshold)
    df = df.with_columns(pl.col("z_score_abs").max().over('ticker').alias('max_z_score_abs_ticker'))
    df = df.filter(pl.col("z_score_abs")==pl.col("max_z_score_abs_ticker"))
    #df = df.group_by(['date', 'ticker']).agg(pl.col(STATISTIC_COL).max().alias("selected_ticker"))

    df = df.with_columns(pl.lit(date).alias('date'))
    dfs.append(df)

In [ ]:
df = pl.concat(dfs)
df.write_parquet("/Volumes/T7/alpha/max_non_rand_10_15_60s_max_ticker.parquet")

# the same but no 0

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import kernel_g
import polars as pl

START_DATE = pd.to_datetime('2023-01-01')
END_DATE = pd.to_datetime('2025-11-15')
date_range = pd.bdate_range(START_DATE, END_DATE)
STATISTIC_COL = 'max_z_score_abs'

bad_dates = []
inputs = []
dfs = []
for date in tqdm(date_range):
    #print(date)
    is_there_trade_file = Path(f"/Volumes/T7/data/trades/time_partioned/{date.strftime('%Y%m%d')}.parquet").exists()
    if not is_there_trade_file:
        continue
    df = pl.read_parquet(f"/Volumes/T7/alpha/10_15_60s_full/{date.strftime('%Y%m%d')}.parquet")

    df_r = pl.read_parquet(f"/Volumes/T7/alpha/10_15_60s_full/{date.strftime('%Y%m%d')}_R.parquet")

    df = kernel_g.kernel_stats_pl(df)
    f_unique = df.unique(subset=['kernel'])
    f_unique = f_unique.to_pandas()

    r = kernel_g.kernel_stats_pl(df_r)
    r_unique = r.unique(subset=['kernel'])
    r_unique = r_unique.to_pandas()

    lower_threshold = r_unique[STATISTIC_COL].quantile(0.99)

    f_unique = f_unique[f_unique[STATISTIC_COL]>lower_threshold]
    df = df.filter(pl.col('grid_mod_n')!=0)
    df = df.filter(pl.col(STATISTIC_COL)>lower_threshold)
    df = df.with_columns(pl.col("z_score_abs").max().over('ticker').alias('max_z_score_abs_ticker'))
    df = df.filter(pl.col("z_score_abs")==pl.col("max_z_score_abs_ticker"))
    #df = df.group_by(['date', 'ticker']).agg(pl.col(STATISTIC_COL).max().alias("selected_ticker"))

    df = df.with_columns(pl.lit(date).alias('date'))
    dfs.append(df)

In [ ]:
df = pl.concat(dfs)
df.write_parquet("/Volumes/T7/alpha/no_0_10_15_60s_max_ticker.parquet")

# Record disagreement

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import kernel_g
import polars as pl

START_DATE = pd.to_datetime('2023-01-01')
END_DATE = pd.to_datetime('2025-11-15')
date_range = pd.bdate_range(START_DATE, END_DATE)
STATISTIC_COL = 'max_z_score_abs'

bad_dates = []
inputs = []
dfs = []
for date in tqdm(date_range):
    #print(date)
    is_there_trade_file = Path(f"/Volumes/T7/data/trades/time_partioned/{date.strftime('%Y%m%d')}.parquet").exists()
    if not is_there_trade_file:
        continue
    df = pl.read_parquet(f"/Volumes/T7/alpha/10_15_60s_full/{date.strftime('%Y%m%d')}.parquet")

    df_r = pl.read_parquet(f"/Volumes/T7/alpha/10_15_60s_full/{date.strftime('%Y%m%d')}_R.parquet")

    df = kernel_g.kernel_stats_pl(df)
    f_unique = df.unique(subset=['kernel'])
    f_unique = f_unique.to_pandas()

    r = kernel_g.kernel_stats_pl(df_r)
    r_unique = r.unique(subset=['kernel'])
    r_unique = r_unique.to_pandas()

    lower_threshold = r_unique[STATISTIC_COL].quantile(0.99)

    f_unique = f_unique[f_unique[STATISTIC_COL]>lower_threshold]

    df = df.filter(pl.col(STATISTIC_COL)>lower_threshold)
    df = df.with_columns(pl.col("z_score_abs").max().over('ticker').alias('max_z_score_abs_ticker'))
    df = df.with_columns(pl.col("z_score").max().over('ticker').alias('max_z_score_ticker'))
    df = df.with_columns(pl.col("z_score").min().over('ticker').alias('min_z_score_ticker'))

    df = df.filter(pl.col("z_score_abs")==pl.col("max_z_score_abs_ticker"))
    #df = df.group_by(['date', 'ticker']).agg(pl.col(STATISTIC_COL).max().alias("selected_ticker"))

    df = df.with_columns(pl.lit(date).alias('date'))
    dfs.append(df)

In [ ]:
df = pl.concat(dfs)
df.write_parquet("/Volumes/T7/alpha/max_non_rand_10_15_60s_max_ticker.parquet")

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import kernel_g
import polars as pl

START_DATE = pd.to_datetime('2023-01-01')
END_DATE = pd.to_datetime('2025-11-15')
date_range = pd.bdate_range(START_DATE, END_DATE)
STATISTIC_COL = 'max_z_score_abs'

bad_dates = []
inputs = []
dfs = []
for date in tqdm(date_range):
    #print(date)
    is_there_trade_file = Path(f"/Volumes/T7/data/trades/time_partioned/{date.strftime('%Y%m%d')}.parquet").exists()
    if not is_there_trade_file:
        continue
    df = pl.read_parquet(f"/Volumes/T7/alpha/14_15_50_full/{date.strftime('%Y%m%d')}.parquet")

    df_r = pl.read_parquet(f"/Volumes/T7/alpha/14_15_50_full/{date.strftime('%Y%m%d')}_R.parquet")

    df = kernel_g.kernel_stats_pl(df)
    f_unique = df.unique(subset=['kernel'])
    f_unique = f_unique.to_pandas()

    r = kernel_g.kernel_stats_pl(df_r)
    r_unique = r.unique(subset=['kernel'])
    r_unique = r_unique.to_pandas()

    lower_threshold = r_unique[STATISTIC_COL].quantile(0.99)

    f_unique = f_unique[f_unique[STATISTIC_COL]>lower_threshold]

    df = df.filter(pl.col(STATISTIC_COL)>lower_threshold)
    df = df.with_columns(pl.col("z_score_abs").max().over('ticker').alias('max_z_score_abs_ticker'))
    df = df.with_columns(pl.col("z_score").max().over('ticker').alias('max_z_score_ticker'))
    df = df.with_columns(pl.col("z_score").min().over('ticker').alias('min_z_score_ticker'))

    df = df.filter(pl.col("z_score_abs")==pl.col("max_z_score_abs_ticker"))
    #df = df.group_by(['date', 'ticker']).agg(pl.col(STATISTIC_COL).max().alias("selected_ticker"))

    df = df.with_columns(pl.lit(date).alias('date'))
    dfs.append(df)

In [ ]:
df = pl.concat(dfs)
df.write_parquet("/Volumes/T7/alpha/14_15_50_max_ticker.parquet")

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import kernel_g
import polars as pl

START_DATE = pd.to_datetime('2023-01-01')
END_DATE = pd.to_datetime('2025-11-15')
date_range = pd.bdate_range(START_DATE, END_DATE)
STATISTIC_COL = 'max_z_score_abs'

bad_dates = []
inputs = []
dfs = []
for date in tqdm(date_range):
    #print(date)
    is_there_trade_file = Path(f"/Volumes/T7/data/trades/time_partioned/{date.strftime('%Y%m%d')}.parquet").exists()
    if not is_there_trade_file:
        continue
    df = pl.read_parquet(f"/Volumes/T7/alpha/10_15_alt_60/{date.strftime('%Y%m%d')}.parquet")

    df_r = pl.read_parquet(f"/Volumes/T7/alpha/10_15_alt_60/{date.strftime('%Y%m%d')}_R.parquet")

    df = kernel_g.kernel_stats_pl(df)
    f_unique = df.unique(subset=['kernel'])
    f_unique = f_unique.to_pandas()

    r = kernel_g.kernel_stats_pl(df_r)
    r_unique = r.unique(subset=['kernel'])
    r_unique = r_unique.to_pandas()

    lower_threshold = r_unique[STATISTIC_COL].quantile(0.99)

    f_unique = f_unique[f_unique[STATISTIC_COL]>lower_threshold]

    df = df.filter(pl.col(STATISTIC_COL)>lower_threshold)
    df = df.with_columns(pl.col("z_score_abs").max().over('ticker').alias('max_z_score_abs_ticker'))
    df = df.with_columns(pl.col("z_score").max().over('ticker').alias('max_z_score_ticker'))
    df = df.with_columns(pl.col("z_score").min().over('ticker').alias('min_z_score_ticker'))

    df = df.filter(pl.col("z_score_abs")==pl.col("max_z_score_abs_ticker"))
    #df = df.group_by(['date', 'ticker']).agg(pl.col(STATISTIC_COL).max().alias("selected_ticker"))

    df = df.with_columns(pl.lit(date).alias('date'))
    dfs.append(df)

In [ ]:
df = pl.concat(dfs)
df.write_parquet("/Volumes/T7/alpha/10_15_alt_60.parquet")

In [8]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import kernel_g
import polars as pl

START_DATE = pd.to_datetime('2023-01-01')
END_DATE = pd.to_datetime('2025-11-15')
date_range = pd.bdate_range(START_DATE, END_DATE)
STATISTIC_COL = 'max_z_score_abs'

bad_dates = []
inputs = []
dfs = []
count = 0
for date in tqdm(date_range):
    #print(date)
    is_there_trade_file = Path(f"/Volumes/T7/data/trades/time_partioned/{date.strftime('%Y%m%d')}.parquet").exists()
    if not is_there_trade_file:
        continue

    try:
        df = pl.read_parquet(f"/Volumes/T7/alpha/10_15_5s_q_size_only/{date.strftime('%Y%m%d')}.parquet")
        df_r = pl.read_parquet(f"/Volumes/T7/alpha/10_15_5s_q_size_only/{date.strftime('%Y%m%d')}_R.parquet")
    except:
        RuntimeError("no data for this date")
        count = count + 1
        continue

    df = kernel_g.kernel_stats_pl(df)
    f_unique = df.unique(subset=['kernel'])
    f_unique = f_unique.to_pandas()

    r = kernel_g.kernel_stats_pl(df_r)
    r_unique = r.unique(subset=['kernel'])
    r_unique = r_unique.to_pandas()

    lower_threshold = r_unique[STATISTIC_COL].quantile(0.99)

    f_unique = f_unique[f_unique[STATISTIC_COL]>lower_threshold]

    df = df.filter(pl.col(STATISTIC_COL)>lower_threshold)
    df = df.with_columns(pl.col("z_score_abs").max().over('ticker').alias('max_z_score_abs_ticker'))
    df = df.with_columns(pl.col("z_score").max().over('ticker').alias('max_z_score_ticker'))
    df = df.with_columns(pl.col("z_score").min().over('ticker').alias('min_z_score_ticker'))

    df = df.filter(pl.col("z_score_abs")==pl.col("max_z_score_abs_ticker"))
    #df = df.group_by(['date', 'ticker']).agg(pl.col(STATISTIC_COL).max().alias("selected_ticker"))

    df = df.with_columns(pl.lit(date).alias('date'))
    dfs.append(df)

100%|██████████| 750/750 [00:57<00:00, 13.02it/s]


In [9]:
count

377

In [7]:
pd.read_parquet("/Volumes/Elements/kernels/10_15_5s_q_size_only/20250115.parquet")

ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet file size is 0 bytes

In [ ]:
df = pl.concat(dfs)
df.write_parquet("/Volumes/T7/alpha/10_15_5s_q_size_only.parquet")